# Phase 5 — One-Class Ensemble Training
**RealCentric Generator-Agnostic Deepfake Detection**  
SVKM AI/ML HPC Cluster

Generator-agnostic detector trained on **real images only** — never sees fake images during training.

**Three detectors fused:**
- Global Mahalanobis (w=0.40)
- Per-component Mahalanobis (w=0.30)
- Isolation Forest (w=0.30)

**Input:** `/data/mpstme-naman/deepfake_detection/data/features/Z_train.npy` (real only)  
**Output:** `/data/mpstme-naman/deepfake_detection/checkpoints/ensemble.pkl`  
**Estimated time:** 5–15 minutes

## Step 1 — Load Real-Only Features

In [ ]:
import sys, numpy as np
sys.path.insert(0, '/data/mpstme-naman/deepfake_detection')
from pathlib import Path

BASE     = Path('/data/mpstme-naman/deepfake_detection')
FEAT_DIR = BASE / 'data' / 'features'
CKPT_DIR = BASE / 'checkpoints'
RES_DIR  = BASE / 'results'
for d in [CKPT_DIR, RES_DIR]: d.mkdir(parents=True, exist_ok=True)

Z_train = np.load(FEAT_DIR/'Z_train.npy');  y_train = np.load(FEAT_DIR/'y_train.npy')
Z_val   = np.load(FEAT_DIR/'Z_val.npy');    y_val   = np.load(FEAT_DIR/'y_val.npy')

Z_real_train = Z_train[y_train == 0]
Z_real_val   = Z_val[y_val   == 0]

print(f'  Real train : {len(Z_real_train):,} samples')
print(f'  Real val   : {len(Z_real_val):,} samples')
print(f'  Feature dim: {Z_train.shape[1]}')
print('\n  Note: fake images are NEVER used in training.')

## Step 2 — Fit the Ensemble

In [ ]:
from config.config_loader import load_config
from src.models.one_class_ensemble import OneClassEnsemble

cfg = load_config()

class PassThrough:
    """Trivial extractor — features already extracted."""
    def extract_batch(self, arr, **kw): return np.array(arr)
    def extract(self, arr, **kw):       return arr

ensemble = OneClassEnsemble(cfg=cfg, use_augmentation=False,
                            use_isolation_forest=True, target_fpr=0.05)
print('Fitting on real features only...')
ensemble.fit(
    [Z_real_train[i] for i in range(len(Z_real_train))],
    PassThrough(),
    real_val_images=[Z_real_val[i] for i in range(len(Z_real_val))]
)
print(f'  ✓  Fitted   threshold τ = {ensemble.threshold:.4f}')

## Step 3 — Save

In [ ]:
path = str(CKPT_DIR/'ensemble.pkl')
ensemble.save(path)
import os; size = os.path.getsize(path)/1e6
print(f'  ✓  {path}  ({size:.1f} MB)')

## Step 4 — Evaluate on CelebDF Test Set

In [ ]:
Z_test = np.load(FEAT_DIR/'Z_test.npy'); y_test = np.load(FEAT_DIR/'y_test.npy')
m = ensemble.evaluate(Z_test[y_test==0], Z_test[y_test==1])
print('=' * 50)
print('  CelebDF Test Set Results')
print('=' * 50)
print(f'  Accuracy  : {m["accuracy"]:.2f}%')
print(f'  AUC       : {m["auc"]:.4f}')
print(f'  TPR       : {m["tpr"]:.4f}  (detection rate)')
print(f'  FPR       : {m["fpr"]:.4f}  (false alarm rate)')
print(f'  F1        : {m["f1"]:.4f}')
print(f'\n  TP={m["tp"]}  FP={m["fp"]}  FN={m["fn"]}  TN={m["tn"]}')

## Step 5 — Cross-Dataset Generalisation

In [ ]:
from src.evaluation.benchmark import _roc_auc
print('Cross-dataset (unseen generators):')
print('=' * 55)
for name, zf, yf in [('FaceForensics++','Z_ff.npy','y_ff.npy'),('Stable Diffusion','Z_sd.npy','y_sd.npy')]:
    Z = np.load(FEAT_DIR/zf); y = np.load(FEAT_DIR/yf)
    scores = ensemble.score_features(Z)
    if len(np.unique(y)) < 2:
        pct = (scores > ensemble.threshold).mean()*100
        print(f'  {name:<20} detection_rate={pct:.1f}%  (fake-only)')
    else:
        m = ensemble.evaluate(Z[y==0], Z[y==1])
        print(f'  {name:<20} ACC={m["accuracy"]:.2f}%  AUC={m["auc"]:.4f}  F1={m["f1"]:.4f}')

## Step 6 — Anomaly Score Distribution

In [ ]:
import matplotlib.pyplot as plt
Z_test = np.load(FEAT_DIR/'Z_test.npy'); y_test = np.load(FEAT_DIR/'y_test.npy')
sr = ensemble.score_features(Z_test[y_test==0])
sf = ensemble.score_features(Z_test[y_test==1])

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(sr, bins=80, alpha=0.6, color='steelblue', label='Real', density=True)
ax.hist(sf, bins=80, alpha=0.6, color='crimson',   label='Fake', density=True)
ax.axvline(ensemble.threshold, color='black', linestyle='--', linewidth=1.8,
           label=f'Threshold τ={ensemble.threshold:.3f}')
ax.set_xlabel('Anomaly Score (higher = more likely fake)')
ax.set_ylabel('Density'); ax.set_title('One-Class Ensemble: Score Distribution')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
out = RES_DIR/'ensemble_score_distribution.png'
plt.savefig(out, dpi=120, bbox_inches='tight'); plt.show()
print(f'  ✓  {out}')

## ✅ Phase 5 Complete

**Saved:** `checkpoints/ensemble.pkl`

**Next:** Open `05_train_autoencoder.ipynb`